# Multi-Query Attention (MQA)

**The First Approach To Solve KV Cache Memory Issue**

Standard multi-head attention requires unique K, V projections for each attention head, which increases the KV cache size proportionally to the number of heads. Multi-Query Attention addresses this by:

- Using a **single shared K, V projection** across all attention heads
- Maintaining **separate Q projections** for each head

This significantly reduces memory requirements during inference while maintaining most of the model's capacity.

### MQA — Multi-Query Attention

**Multi-Query Attention (MQA)** is a variant of **Multi-Head Attention** designed to make LLM inference faster and use less memory.

The key idea is simple:

> **All attention heads share the same Key (K) and Value (V), while each head still has its own Query (Q).**

### Standard Multi-Head Attention (MHA)

Each head has its own:

```text
Q₁  ─┐ ──> Its Own K, V
Q₂  ─┤ ──> Its Own K, V
Q₃  ─┤ ──> Its Own K, V
... ─┤ ──> Its Own K, V
Qₕ  ─┘ ──> Its Own K, V
```

So during autoregressive generation, the **KV cache becomes large** because we must store K and V for every head.

### Multi-Query Attention (MQA)

MQA uses:

```text
Q₁  ─┐
Q₂  ─┤
Q₃  ─┤──> Shared K, V
... ─┤
Qₕ  ─┘
```

There are still multiple **Q heads**, but only **one K head and one V head**.

## Listing 2.3: Implementing Multi-Query Attention (MQA)

The code below implements a Multi-Query Attention layer from scratch. Notice the key differences from standard multi-head attention:

1. Query projection (`self.W_q`) maps to the full model dimension (`d_model`)
2. Key and value projections (`self.W_k` and `self.W_v`) map to just a single head dimension (`self.d_head`)
3. We use `repeat()` to duplicate the single key and value for all query heads

This implementation clearly shows how MQA reduces the parameter count and memory footprint during inference, especially for the KV cache which only needs to store a single key-value pair per token instead of one per attention head.

In [ ]:
import torch
from torch import nn

# ====================================================
# LISTING 2.3: Implementing an MQA layer from scratch
# ====================================================
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, self.d_head) # Single projection for K
        self.W_v = nn.Linear(d_model, self.d_head) # Single projection for V
        self.W_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        
        # Using a fixed size mask for demonstration. A dynamic one is better in practice.
        self.register_buffer('mask', torch.triu(torch.ones(1, 1, 1024, 1024), diagonal=1))

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        # Query: (B, num_heads, seq_len, d_head)
        q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # Key & Value: (B, 1, seq_len, d_head)
        k = self.W_k(x).view(batch_size, seq_len, 1, self.d_head).transpose(1, 2)
        v = self.W_v(x).view(batch_size, seq_len, 1, self.d_head).transpose(1, 2)

        # Repeat K and V for each query head
        k = k.repeat(1, self.num_heads, 1, 1) # (B, num_heads, seq_len, d_head)
        v = v.repeat(1, self.num_heads, 1, 1) # (B, num_heads, seq_len, d_head)

        attn_scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)

        # Apply causal mask
        attn_scores = attn_scores.masked_fill(self.mask[:,:,:seq_len,:seq_len] == 1, float('-inf'))

        attn_weights = torch.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vector = (attn_weights @ v).transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        output = self.W_o(context_vector)
        return output

# -----------------------------------------------
# ---------------- Usage Example ----------------
# -----------------------------------------------
d_model = 512
num_heads = 8
batch_size = 4
seq_len = 64

mqa_layer = MultiQueryAttention(d_model, num_heads)
dummy_input = torch.randn(batch_size, seq_len, d_model)
output = mqa_layer(dummy_input)

print("✅ MQA Layer successful! 🎉")
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")

✅ MQA Layer successful! 🎉
Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])


### Why is MQA useful?

The biggest benefit is **KV-cache reduction**.

For example, with **32 attention heads**:

* MHA → 32 K heads + 32 V heads
* MQA → **1 K head + 1 V head**

So the KV cache can become roughly **32× smaller**.

This means:

* 🚀 Faster autoregressive inference
* 💾 Much lower KV-cache memory usage
* 👥 More concurrent requests can fit in GPU memory
* ⚡ Particularly useful for long-context LLM serving

### MHA vs MQA

|             | MHA    | MQA              |
| ----------- | ------ | ---------------- |
| Query heads | Many   | Many             |
| Key heads   | Many   | **1**            |
| Value heads | Many   | **1**            |
| KV Cache    | Large  | **Much smaller** |
| Inference   | Slower | **Faster**       |

**In one sentence:**

> **MQA keeps multiple Query heads but shares a single Key and Value across all heads, dramatically reducing KV-cache memory and improving LLM inference efficiency.**


### The Dark Side of MQA

The main downside of **MQA** is that sharing the same **K and V across all attention heads reduces the model's ability to learn different representations for each head**.

In short:

> **MQA saves a lot of memory and speeds up inference, but can slightly hurt model quality compared with full MHA.**

So the trade-off is:

**🚀 Faster + 💾 Less memory → potentially 📉 Lower quality**

**MQA** Results is **Significant Performance Degradation** compared to **MHA**:

> we know that each Head in **MHA** has its own K and V projections, which allows the model to learn different perspectives for each head. While **MQA** shares the same K and V across all heads, this can limit the model's ability to capture diverse patterns in the data.